## 1. Environment Setup

### 1.1. Install Required Libraries

In [72]:
# Install required packages

# !pip install numpy pandas nltk scikit-learn matplotlib seaborn wordcloud

# Download NLTK data

import nltk

nltk.download("brown")
nltk.download("stopwords")
nltk.download('universal_tagset')
nltk.download('punkt')
nltk.download('reuters')

[nltk_data] Downloading package brown to
[nltk_data]     C:\Users\ngovi\AppData\Roaming\nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ngovi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package universal_tagset to
[nltk_data]     C:\Users\ngovi\AppData\Roaming\nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ngovi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package reuters to
[nltk_data]     C:\Users\ngovi\AppData\Roaming\nltk_data...
[nltk_data]   Package reuters is already up-to-date!


True

### 1.2. Import Libraries

Import all necessary libraries for your implementation.

In [18]:
# Import all necessary libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

# Set random seed for reproducibility
np.random.seed(42)

## 2. Autocorrect System


### 2.1 Vocabulary Building

In [3]:
from collections import Counter
from nltk.corpus import brown


def build_vocabulary():
    print("Loading corpus...")
    # 1. Get the raw list from NLTK
    raw_tokens = brown.words()

    # 2. Process into a list of clean, lowercase words
    # We filter out non-alphabetic tokens to remove punctuation like ',' or '!'
    clean_word_list = [word.lower() for word in raw_tokens if word.isalpha()]

    # 3. Convert the list into a Dictionary of counts
    # Counter is a subclass of dict specifically for counting hashable objects
    vocab_counts = Counter(clean_word_list)

    print(f"Total words in corpus: {len(clean_word_list)}")
    print(f"Unique words in vocabulary: {len(vocab_counts)}")

    return vocab_counts


vocab = build_vocabulary()
vocab

Loading corpus...
Total words in corpus: 981716
Unique words in vocabulary: 40234


Counter({'the': 69971,
         'of': 36412,
         'and': 28853,
         'to': 26158,
         'a': 23195,
         'in': 21337,
         'that': 10594,
         'is': 10109,
         'was': 9815,
         'he': 9548,
         'for': 9489,
         'it': 8760,
         'with': 7289,
         'as': 7253,
         'his': 6996,
         'on': 6741,
         'be': 6377,
         'at': 5372,
         'by': 5306,
         'i': 5164,
         'this': 5145,
         'had': 5133,
         'not': 4610,
         'are': 4394,
         'but': 4381,
         'from': 4370,
         'or': 4206,
         'have': 3942,
         'an': 3740,
         'they': 3620,
         'which': 3561,
         'one': 3292,
         'you': 3286,
         'were': 3284,
         'her': 3036,
         'all': 3001,
         'she': 2860,
         'there': 2728,
         'would': 2714,
         'their': 2669,
         'we': 2652,
         'him': 2619,
         'been': 2472,
         'has': 2437,
         'when': 2331,
   

### 2.2 Probability Calculation

In [4]:
def get_probs(vocab_counts):
    total_count = sum(vocab_counts.values())
    word_probs = {word: count / total_count for word, count in vocab_counts.items()}
    return word_probs

word_probabilities = get_probs(vocab)
word_probabilities

{'the': 0.07127417705324146,
 'fulton': 1.731661702569786e-05,
 'county': 0.00015788680229312753,
 'grand': 4.889397748432337e-05,
 'jury': 6.824784357186804e-05,
 'said': 0.001997522705140794,
 'friday': 6.111747185540421e-05,
 'an': 0.0038096557456535293,
 'investigation': 5.194985107709358e-05,
 'of': 0.03709015641998297,
 'recent': 0.00018233379103528924,
 'primary': 9.778795496864674e-05,
 'election': 7.843408888110206e-05,
 'produced': 9.167620778310632e-05,
 'no': 0.00217883787164516,
 'evidence': 0.00020779940430837432,
 'that': 0.010791308280602537,
 'any': 0.0013690313695610542,
 'irregularities': 8.148996247387228e-06,
 'took': 0.0004339340501733699,
 'place': 0.00058061598262634,
 'further': 0.00022206014774130197,
 'in': 0.02173439161631266,
 'presentments': 1.0186245309234035e-06,
 'city': 0.0004003194406528976,
 'executive': 5.6024349200787194e-05,
 'committee': 0.00017112892119513178,
 'which': 0.0036273219546182397,
 'had': 0.00522859971722983,
 'charge': 0.00012427219

### 2.3 Levenshtein Distance

In [5]:
def min_edit_distance(source, target, ins_cost=1, del_cost=1, sub_cost=1):
    m = len(source)
    n = len(target)

    # Create a distance matrix
    dp = np.zeros((m + 1, n + 1), dtype=int)

    # Initialize the first row and column
    for i in range(m + 1):
        dp[i][0] = i * del_cost
    for j in range(n + 1):
        dp[0][j] = j * ins_cost

    # Fill the distance matrix
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if source[i - 1] == target[j - 1]:
                cost = 0
            else:
                cost = sub_cost

            dp[i][j] = min(
                dp[i - 1][j] + del_cost,  # Deletion
                dp[i][j - 1] + ins_cost,  # Insertion
                dp[i - 1][j - 1] + cost,  # Substitution
            )

    return dp[m][n]

In [6]:
# Testing the min_edit_distance function
from nltk import edit_distance

assert min_edit_distance("apple", "aple") == edit_distance("apple", "aple")

### 2.4 Candidate Generation

In [7]:
def edit_one_letter(word):
    letters = 'abcdefghijklmnopqrstuvwxyz'
    edits = set()

    # Deletions
    for i in range(len(word)):
        edits.add(word[:i] + word[i+1:])

    # Transpositions
    for i in range(len(word) - 1):
        edits.add(word[:i] + word[i+1] + word[i] + word[i+2:])

    # Alterations
    for i in range(len(word)):
        for l in letters:
            edits.add(word[:i] + l + word[i+1:])

    # Insertions
    for i in range(len(word) + 1):
        for l in letters:
            edits.add(word[:i] + l + word[i:])

    return edits

# Example usage
edits = edit_one_letter("cat")
print(f"Possible edits for 'cat': {edits}")

def edit_two_letters(word):
    edits = set()
    one_letter_edits = edit_one_letter(word)
    for edit in one_letter_edits:
        edits.update(edit_one_letter(edit))
    return edits

# Example usage
edits_two = edit_two_letters("cat")
print(f"Possible two-letter edits for 'cat': {edits_two}")

Possible edits for 'cat': {'caz', 'cht', 'nat', 'cagt', 'catq', 'cart', 'cqat', 'clat', 'capt', 'cit', 'catj', 'ctt', 'caa', 'catt', 'cct', 'vat', 'cau', 'tat', 'mcat', 'cati', 'jat', 'car', 'xcat', 'cyat', 'cmat', 'cah', 'cdt', 'mat', 'fcat', 'oat', 'caqt', 'ncat', 'cas', 'catg', 'crat', 'lcat', 'cbt', 'cawt', 'cet', 'cats', 'cxat', 'catb', 'ctat', 'clt', 'cnat', 'cai', 'cate', 'cajt', 'cfat', 'zat', 'cqt', 'catz', 'eat', 'czt', 'iat', 'at', 'bcat', 'catk', 'rcat', 'caw', 'cato', 'ocat', 'cot', 'cax', 'rat', 'wat', 'acat', 'catv', 'catn', 'catf', 'hcat', 'dcat', 'tcat', 'cdat', 'cam', 'cpt', 'jcat', 'sat', 'scat', 'caft', 'bat', 'cait', 'cao', 'yat', 'vcat', 'cakt', 'cbat', 'cjat', 'czat', 'cata', 'ckt', 'catd', 'gcat', 'ycat', 'cant', 'camt', 'cast', 'fat', 'pat', 'cae', 'cat', 'gat', 'cayt', 'xat', 'pcat', 'cad', 'cwat', 'caot', 'qat', 'ecat', 'catu', 'dat', 'ct', 'catp', 'catl', 'lat', 'ccat', 'cal', 'aat', 'cay', 'kcat', 'cuat', 'kat', 'cap', 'cyt', 'act', 'icat', 'chat', 'catc', 

### 2.5 Ranking

In [8]:
def get_corrections(word, probs, vocab, n=5):
    suggestions = {}

    # Tier 1: Is the word already valid? (0 edits)
    if word in vocab:
        suggestions[word] = probs.get(word, 0)
        print(f"'{word}' found in vocabulary.")

    # Tier 2: 1-letter edits
    if not suggestions:
        suggestions = {w: probs.get(w, 0) for w in edit_one_letter(word) if w in vocab}
        if suggestions:
            print(f"Found {len(suggestions)} candidates at edit distance 1.")

    # Tier 3: 2-letter edits
    if not suggestions:
        suggestions = {w: probs.get(w, 0) for w in edit_two_letters(word) if w in vocab}
        if suggestions:
            print(f"Found {len(suggestions)} candidates at edit distance 2.")

    # Tier 4: Failsafe (return original)
    if not suggestions:
        suggestions[word] = 0
        print("No corrections found. Returning original.")

    # Sort by probability (Descending)
    ranked_suggestions = sorted(
        suggestions.items(), key=lambda item: item[1], reverse=True
    )

    return ranked_suggestions[:n]


# Example usage
top_corrections = 5
word_to_check = "Doo"
corrections = get_corrections(word_to_check, word_probabilities, vocab, n=top_corrections)
print(f"Top {top_corrections} corrections for '{word_to_check}': {corrections}")

Found 5 candidates at edit distance 1.
Top 5 corrections for 'Doo': [('too', 0.0008495328587901185), ('zoo', 9.167620778310631e-06), ('woo', 3.0558735927702106e-06), ('doo', 2.037249061846807e-06), ('boo', 1.0186245309234035e-06)]


In [9]:
def correct_sentence(sentence, vocab, probs):
    # 1. Split the sentence into words
    words = sentence.strip().split()
    corrected_words = []

    print(f"Original: {sentence}")
    print("=" * 40)

    for word in words:
        # 2. Lowercase the word for the lookup (since our vocab is likely lowercase)
        word_lower = word.lower()

        # 3. Get the top correction (n=1)
        top_corrections = get_corrections(word_lower, probs, vocab, n=1)

        # 4. Extract the word string from the (word, prob) tuple
        if top_corrections:
            best_word = top_corrections[0][0]
        else:
            best_word = word  # Fallback

        corrected_words.append(best_word)

        if word_lower != best_word:
            print(f"Corrected: {word} -> {best_word}")

        print("-" * 40)

    # 5. Rejoin the sentence
    return " ".join(corrected_words)


# --- Run the Test ---
input_sentence = "The jury saiid no evidnce was founnd"
result = correct_sentence(input_sentence, vocab, word_probabilities)

print("-" * 40)
print(f"Final:    {result}")

Original: The jury saiid no evidnce was founnd
'the' found in vocabulary.
----------------------------------------
'jury' found in vocabulary.
----------------------------------------
Found 1 candidates at edit distance 1.
Corrected: saiid -> said
----------------------------------------
'no' found in vocabulary.
----------------------------------------
Found 1 candidates at edit distance 1.
Corrected: evidnce -> evidence
----------------------------------------
'was' found in vocabulary.
----------------------------------------
Found 1 candidates at edit distance 1.
Corrected: founnd -> found
----------------------------------------
----------------------------------------
Final:    the jury said no evidence was found


## 3. POS Tagging with HMM


### 3.1 Data Preparation & Vocabulary Handling

In [10]:
def get_data_split(train_ratio=0.8):
    # Load the Brown corpus sentences
    sentences = brown.tagged_sents(tagset="universal")
    total_sentences = len(sentences)
    train_size = int(total_sentences * train_ratio)

    # Split into training and testing sets
    train_sentences = sentences[:train_size]
    test_sentences = sentences[train_size:]

    return train_sentences, test_sentences


def build_vocab(train_data, min_freq=2):
    """
    Builds a vocab dictionary.
    Words with frequency < min_freq are treated as unknown.
    """
    # Flatten the data to get all words
    all_words = [word.lower() for sentence in train_data for word, tag in sentence]
    word_counts = Counter(all_words)

    # specific vocab list handling unknown
    vocab = {word for word, count in word_counts.items() if count >= min_freq}

    return vocab


# --- Execution ---
train_data, test_data = get_data_split()
vocab = build_vocab(train_data)

print(f"Training Sentences: {len(train_data)}")
print(f"Vocabulary Size: {len(vocab)}")
print(f"Sample Sentence: {train_data[0]}")

Training Sentences: 45872
Vocabulary Size: 25347
Sample Sentence: [('The', 'DET'), ('Fulton', 'NOUN'), ('County', 'NOUN'), ('Grand', 'ADJ'), ('Jury', 'NOUN'), ('said', 'VERB'), ('Friday', 'NOUN'), ('an', 'DET'), ('investigation', 'NOUN'), ('of', 'ADP'), ("Atlanta's", 'NOUN'), ('recent', 'ADJ'), ('primary', 'NOUN'), ('election', 'NOUN'), ('produced', 'VERB'), ('``', '.'), ('no', 'DET'), ('evidence', 'NOUN'), ("''", '.'), ('that', 'ADP'), ('any', 'DET'), ('irregularities', 'NOUN'), ('took', 'VERB'), ('place', 'NOUN'), ('.', '.')]


### 3.2 Calculate Matrices

#### 3.2.1 Create Dictionaries

In [14]:
def create_dictionaries(train_data, vocab):
    """
    Creates emission_counts, transition_counts, and tag_counts.
    """
    emission_counts = Counter()
    transition_counts = Counter()
    tag_counts = Counter()

    for sentence in train_data:
        previous_tag = "--s--"  # Start token
        tag_counts[previous_tag] += 1

        for word, tag in sentence:
            # 1. Update tag counts
            tag_counts[tag] += 1

            # 2. Update transition counts (previous_tag -> current_tag)
            transition_counts[(previous_tag, tag)] += 1

            # 3. Update emission counts (tag -> word)
            if word.lower() in vocab:
                emission_counts[(tag, word.lower())] += 1
            else:
                emission_counts[(tag, "--unk--")] += 1

            previous_tag = tag

    return emission_counts, transition_counts, tag_counts


# --- Execution ---
emission_counts, transition_counts, tag_counts = create_dictionaries(train_data, vocab)
print(f"Sample Tag Counts: {list(tag_counts.items())[:5]}")
print(f"Sample Transition Counts: {list(transition_counts.items())[:5]}")
print(f"Sample Emission Counts: {list(emission_counts.items())[:5]}")

Sample Tag Counts: [('--s--', 45872), ('DET', 116989), ('NOUN', 241528), ('ADJ', 73866), ('VERB', 150459)]
Sample Transition Counts: [(('--s--', 'DET'), 10665), (('DET', 'NOUN'), 72414), (('NOUN', 'NOUN'), 37458), (('NOUN', 'ADJ'), 3198), (('ADJ', 'NOUN'), 49062)]
Sample Emission Counts: [(('DET', 'the'), 61188), (('NOUN', 'fulton'), 17), (('NOUN', 'county'), 149), (('ADJ', 'grand'), 41), (('NOUN', 'jury'), 60)]


#### 3.2.2 Calculate Matrices

In [15]:
def calculate_matrices(
    emission_counts, transition_counts, tag_counts, vocab, alpha=0.001
):
    """
    Calculates A (Transition) and B (Emission) probability matrices.
    """
    all_tags = sorted(tag_counts.keys())

    # --- Transition Matrix A ---
    # Rows: previous tags, Columns: current tags
    A = np.zeros((len(all_tags), len(all_tags)))

    for i, prev_tag in enumerate(all_tags):
        # Calculate the total transitions FROM this tag
        total_transitions_from_prev = 0
        for t in all_tags:
            total_transitions_from_prev += transition_counts.get((prev_tag, t), 0)

        for j, curr_tag in enumerate(all_tags):
            count = transition_counts.get((prev_tag, curr_tag), 0)

            # Denominator: transitions observed + smoothing
            denominator = total_transitions_from_prev + (alpha * len(all_tags))

            # Avoid division by zero (if a tag only appears at the very end of sentences)
            if denominator == 0:
                A[i, j] = 0
            else:
                A[i, j] = (count + alpha) / denominator

    # --- Emission Matrix B ---
    # Rows: tags, Columns: words in vocab + --unk--

    B = {}  # dictionary mapping (tag, word) -> probability
    vocab_list = sorted(list(vocab)) + ["--unk--"]

    for tag in all_tags:
        denominator = tag_counts[tag] + (alpha * len(vocab_list))

        for word in vocab_list:
            count = emission_counts.get((tag, word), 0)

            # apply Laplace smoothing
            B[(tag, word)] = (count + alpha) / denominator

    return A, B, all_tags


# --- Execution ---
A, B, all_tags = calculate_matrices(
    emission_counts, transition_counts, tag_counts, vocab
)

In [ ]:
# Verification of the matrices
# def verify_matrices(A, B, all_tags, vocab_list):
#     print("--- 1. Mathematical Verification ---")

#     # Check Transition Matrix Row Sums
#     row_sums = np.sum(A, axis=1)
#     print(f"Transition Matrix Row Sums (First 5): {row_sums[:5]}")
#     is_valid_A = np.allclose(row_sums, 1.0, atol=1e-5)
#     print(f"Matrix A is valid (rows sum to 1): {is_valid_A}")

#     # Check Emission Matrix for a specific tag (e.g., 'NOUN')
#     # We sum P(word | NOUN) for all words in vocab + <UNK>
#     test_tag = "NOUN"
#     emission_sum = 0
#     for word in vocab_list:
#         emission_sum += B.get((test_tag, word), 0)

#     print(f"Emission Sum for '{test_tag}': {emission_sum:.5f}")

#     print("\n--- 2. Linguistic Verification ---")

#     # Helper to get index
#     tag_map = {tag: i for i, tag in enumerate(all_tags)}

#     # Check: What follows a Determiner (DET)?
#     if "DET" in tag_map and "NOUN" in tag_map:
#         det_idx = tag_map["DET"]
#         noun_idx = tag_map["NOUN"]
#         verb_idx = tag_map["VERB"] if "VERB" in tag_map else -1

#         prob_noun_after_det = A[det_idx, noun_idx]
#         prob_verb_after_det = A[det_idx, verb_idx]

#         print(f"P(NOUN | DET): {prob_noun_after_det:.4f} (Expected: High)")
#         print(f"P(VERB | DET): {prob_verb_after_det:.4f} (Expected: Low)")

#     # Check: Emission for 'the' under DET vs NOUN
#     prob_the_det = B.get(("DET", "the"), 0)
#     prob_the_noun = B.get(("NOUN", "the"), 0)
#     print(f"P('the' | DET):  {prob_the_det:.4f} (Expected: High)")
#     print(f"P('the' | NOUN): {prob_the_noun:.4f} (Expected: Low/Zero)")


# # --- Visualization (Heatmap) ---
# def plot_transition_heatmap(A, all_tags):
#     plt.figure(figsize=(10, 8))
#     sns.heatmap(A, xticklabels=all_tags, yticklabels=all_tags, cmap="Blues")
#     plt.title("Transition Probability Matrix (A)")
#     plt.xlabel("Next Tag")
#     plt.ylabel("Previous Tag")
#     plt.show()


# # --- Run the tests ---
# # Re-create vocab list including UNK for the math check
# vocab_list = sorted(list(vocab)) + ["<UNK>"]
# verify_matrices(A, B, all_tags, vocab_list)
# plot_transition_heatmap(A, all_tags)

### 3.3 Viterbi Algorithm

In [19]:
def viterbi(words, A, B, all_tags, vocab):
    """
    Decodes the best path of tags for a given sequence of words.
    """
    num_tags = len(all_tags)
    num_words = len(words)

    # best_probs[i, j] = log prob of the best path to word i ending in tag j
    # Initialize with -infinity (log(0))
    best_probs = np.full((num_words, num_tags), -np.inf)

    # best_paths[i, j] = index of the previous tag that gave the best prob
    best_paths = np.zeros((num_words, num_tags), dtype=int)

    # map tags to indices for easier access
    tag_to_index = {tag: i for i, tag in enumerate(all_tags)}

    # --- Step 1: Handle the first word ---
    # Check if first word is in vocab, else use UNK
    first_word = words[0].lower() if words[0].lower() in vocab else "--unk--"

    # find index of start token "--s--"
    if "--s--" in tag_to_index:
        start_tag_index = tag_to_index["--s--"]
    else:
        raise ValueError("Start tag '--s--' not found in tag list.")

    for i, tag in enumerate(all_tags):
        # 1. Transition Probability: P(Tag | Start)
        # use a small epsilon 1e-10 to avoid log(0) errors
        epsilon = 1e-10
        trans_prob = math.log(A[start_tag_index, i] + epsilon)

        # 2. Emission Probability: P(Word | Tag)
        # If word is not in vocab, use emission probability for UNK
        emit_prob = math.log(B.get((tag, first_word), epsilon))
        # Update best_probs for first word
        best_probs[0, i] = trans_prob + emit_prob

    # --- Step 2: Forward Pass ---
    for i in range(1, num_words):
        current_word = words[i].lower() if words[i].lower() in vocab else "--unk--"

        for curr_tag_idx, curr_tag in enumerate(all_tags):
            best_prob_for_tag = -float("inf")
            best_prev_tag_idx = -1

            emission_prob = math.log(B.get((curr_tag, current_word), epsilon))

            for prev_tag_idx, prev_tag in enumerate(all_tags):
                # Path calculation:
                # Prob = (Prob of path to Prev Tag) + (Transition Prev->Curr) + (Emission Curr)

                trans_prob = math.log(A[prev_tag_idx, curr_tag_idx] + epsilon)

                # prob from previous step (i-1)
                prev_path_prob = best_probs[i - 1, prev_tag_idx]

                total_prob = prev_path_prob + trans_prob + emission_prob

                if total_prob > best_prob_for_tag:
                    best_prob_for_tag = total_prob
                    best_prev_tag_idx = prev_tag_idx

            # Store the winner
            best_probs[i, curr_tag_idx] = best_prob_for_tag
            best_paths[i, curr_tag_idx] = best_prev_tag_idx

    return best_probs, best_paths, tag_to_index

### 3.4 Backtracking

In [20]:
def viterbi_backward(best_probs, best_paths, all_tags, tag_map):
    """
    Reconstructs the best path sequence from the matrices.
    """
    num_words = best_probs.shape[0]

    # 1. Find the best FINAL tag (the one with highest prob in the last column)
    z = [None] * num_words

    # Get index of max value in the last row of best_probs
    best_last_tag_idx = np.argmax(best_probs[num_words - 1, :])
    z[num_words - 1] = best_last_tag_idx

    # 2. Walk backwards
    for i in range(num_words - 1, 0, -1):
        # look up the best previous tag from best_paths
        z[i - 1] = best_paths[i, z[i]]

    # 3. Convert indices back to tags
    index_to_tag = {i: tag for tag, i in tag_map.items()}
    predicted_tags = [index_to_tag[idx] for idx in z]

    return predicted_tags

In [29]:
def tag_sentence(sentence, A, B, all_tags, vocab):
    words = sentence.split()
    best_probs, best_paths, tag_map = viterbi(words, A, B, all_tags, vocab)
    predicted_tags = viterbi_backward(best_probs, best_paths, all_tags, tag_map)
    
    # Zip them together for display
    return list(zip(words, predicted_tags))

# --- Execution ---
test_sent = "The market is volatile today"
# test_sent = "Hello there general Kenobi"
result = tag_sentence(test_sent, A, B, all_tags, vocab)

print(f"Sentence: {test_sent}")
print(f"Tags: {result}")

Sentence: The market is volatile today
Tags: [('The', 'DET'), ('market', 'NOUN'), ('is', 'VERB'), ('volatile', 'ADJ'), ('today', 'NOUN')]


### 3.5 Evaluate Accuracy

In [30]:
def compute_accuracy(test_data, A, B, all_tags, vocab):
    """
    Runs the Viterbi algorithm on the test set and calculates accuracy.
    """
    num_correct = 0
    total_tags = 0

    # Iterate over every sentence in the test set
    for i, sentence in enumerate(test_data):
        # 1. Split into words and ground-truth tags
        words = [pair[0] for pair in sentence]
        true_tags = [pair[1] for pair in sentence]

        # 2. Run Viterbi
        # Note: We use the functions you just wrote
        best_probs, best_paths, tag_map = viterbi(words, A, B, all_tags, vocab)
        pred_tags = viterbi_backward(best_probs, best_paths, all_tags, tag_map)

        # 3. Compare prediction vs truth
        # Safely zip in case lengths mismatch (rare but possible with bugs)
        for true, pred in zip(true_tags, pred_tags):
            if true == pred:
                num_correct += 1
            total_tags += 1

        # Optional: Progress print every 500 sentences
        if (i + 1) % 500 == 0:
            print(f"Processed {i + 1} sentences...")

    return num_correct / total_tags


# --- Run the Evaluation ---
print(f"Evaluating on {len(test_data)} sentences. This might take a moment...")
accuracy = compute_accuracy(test_data, A, B, all_tags, vocab)
print(f"Model Accuracy: {accuracy*100:.2f}%")

Evaluating on 11468 sentences. This might take a moment...
Processed 500 sentences...
Processed 1000 sentences...
Processed 1500 sentences...
Processed 2000 sentences...
Processed 2500 sentences...
Processed 3000 sentences...
Processed 3500 sentences...
Processed 4000 sentences...
Processed 4500 sentences...
Processed 5000 sentences...
Processed 5500 sentences...
Processed 6000 sentences...
Processed 6500 sentences...
Processed 7000 sentences...
Processed 7500 sentences...
Processed 8000 sentences...
Processed 8500 sentences...
Processed 9000 sentences...
Processed 9500 sentences...
Processed 10000 sentences...
Processed 10500 sentences...
Processed 11000 sentences...
Model Accuracy: 95.09%


## 4. N-gram Language Model


### 4.1 Data Preparation

In [74]:
from nltk.corpus import reuters


def get_reuters_data():
    """
    Correctly retrieves training and testing data from Reuters corpus
    by filtering file IDs based on their prefix.
    """
    all_fileids = reuters.fileids()
    
    train_ids = [fid for fid in all_fileids if fid.startswith('training/')]
    test_ids = [fid for fid in all_fileids if fid.startswith('test/')]
    
    print(f"Loading {len(train_ids)} training docs and {len(test_ids)} test docs...")
    train_data = reuters.sents(train_ids)
    test_data = reuters.sents(test_ids)
    
    return train_data, test_data


def build_vocab(train_data, min_freq=3):
    """
    Creates a vocabulary of words that appear at least min_freq times.
    Returns a set of valid words.
    """
    flat_words = [w for sent in train_data for w in sent]
    counts = Counter(flat_words)

    # Create set of words >= min_freq
    vocab = {word for word, count in counts.items() if count >= min_freq}
    return vocab


def preprocess_sentences(sentences, vocab, n):
    """
    1. Replaces OOV words with <UNK>
    2. Adds start <s> and end </s> tokens based on n-gram order.
       For Trigram (n=3), we need two start tokens: <s> <s> word...
    """
    processed = []
    unk_token = "<UNK>"
    start_token = "<s>"
    end_token = "</s>"

    for sent in sentences:
        # Replace OOV
        clean_sent = [w.lower() if w.lower() in vocab else unk_token for w in sent]

        # Add Padding
        # For n=3, padding is ['<s>', '<s>']
        padding = [start_token] * (n - 1)

        # Final sentence: <s> <s> w1 w2 ... wn </s>
        clean_sent = padding + clean_sent + [end_token]
        processed.append(clean_sent)

    return processed


# --- Execute Setup ---
train_data_raw, test_data_raw = get_reuters_data()

print(f"Training Sentences: {len(train_data_raw)}")
print(f"Testing Sentences:  {len(test_data_raw)}")

vocab = build_vocab(train_data_raw, min_freq=2)
print(f"Vocab size: {len(vocab)}")

# Prepare for Trigrams (n=3)
train_data_processed = preprocess_sentences(train_data_raw, vocab, n=3)
test_data_processed = preprocess_sentences(test_data_raw, vocab, n=3)
print(f"Sample processed sentence: {train_data_processed[0]}")

Loading 7769 training docs and 3019 test docs...
Training Sentences: 40277
Testing Sentences:  14439
Vocab size: 22030
Sample processed sentence: ['<s>', '<s>', '<UNK>', 'cocoa', 'review', 'showers', 'continued', 'throughout', 'the', 'week', 'in', 'the', '<UNK>', 'cocoa', 'zone', ',', '<UNK>', 'the', 'drought', 'since', 'early', 'january', 'and', 'improving', 'prospects', 'for', 'the', 'coming', 'temporao', ',', 'although', 'normal', 'humidity', 'levels', 'have', 'not', 'been', 'restored', ',', '<UNK>', '<UNK>', 'said', 'in', 'its', 'weekly', 'review', '.', '</s>']


### 4.2 Counting N-Grams

In [75]:
def count_n_grams(data, n):
    """
    Counts n-grams from a list of preprocessed sentences.
    Returns a dictionary: { (w_1, ..., w_n): count }
    """
    n_grams = {}

    for sent in data:
        for i in range(len(sent) - n + 1):
            n_gram = tuple(sent[i : i + n])
            n_grams[n_gram] = n_grams.get(n_gram, 0) + 1

    return n_grams


# --- Test Counting ---
unigrams = count_n_grams(train_data_processed, 1)
bigrams = count_n_grams(train_data_processed, 2)
trigrams = count_n_grams(train_data_processed, 3)

print(f"Unique Unigrams: {len(unigrams)}")
print(f"Unique Bigrams:  {len(bigrams)}")
print(f"Unique Trigrams: {len(trigrams)}")
print(f"Count of ('the',): {unigrams.get(('the',), 0)}")

Unique Unigrams: 10674
Unique Bigrams:  225528
Unique Trigrams: 575808
Count of ('the',): 51405


### 4.3 Probability & Autocomplete

In [76]:
def estimate_probability(
    word, previous_n_gram, n_gram_counts, n_plus1_gram_counts, vocabulary_size, k=1.0
):
    """
    Calculates the probability of 'word' following 'previous_n_gram'.
    """
    previous_n_gram = tuple(previous_n_gram)

    # 1. get counts
    previous_n_gram_count = n_gram_counts.get(previous_n_gram, 0)
    n_plus1_gram = previous_n_gram + (word,)
    n_plus1_gram_count = n_plus1_gram_counts.get(n_plus1_gram, 0)

    # 2. apply Laplace smoothing
    numerator = n_plus1_gram_count + k
    denominator = previous_n_gram_count + k * vocabulary_size

    return numerator / denominator


def suggest_next_word(
    previous_tokens, n_gram_counts, n_plus1_gram_counts, vocabulary, k=1.0
):
    """
    Checks ALL words in the vocab and returns the one with the highest probability.
    """
    # Get the correct history length (N-1)
    n = len(list(n_gram_counts.keys())[0])
    previous_n_gram = tuple(previous_tokens[-n:])

    vocabulary_size = len(vocabulary)

    best_prob = 0
    best_word = None

    # Loop through every known word to find the winner
    for word in vocabulary:
        prob = estimate_probability(
            word,
            previous_n_gram,
            n_gram_counts,
            n_plus1_gram_counts,
            vocabulary_size,
            k,
        )

        if prob > best_prob:
            best_prob = prob
            best_word = word

    return best_word, best_prob


# --- Test Prediction (Trigram Model) ---
# We use 'bigrams' as the history denominator, and 'trigrams' as the target numerator
vocab_list = list(vocab)

next_word, prob = suggest_next_word(
    ["the", "united"],  # The history
    bigrams,  # n_gram_counts
    trigrams,  # n_plus1_gram_counts
    vocab_list,
    k=1.0,
)

print(f"Input: 'the united ...'")
print(f"Predicted: {next_word}")
print(f"Probability: {prob:.4f}")

Input: 'the united ...'
Predicted: states
Probability: 0.0180


In [78]:
def calculate_perplexity(
    sentence, n_gram_counts, n_plus1_gram_counts, vocab_size, k=1.0
):
    """
    Calculates perplexity for a single sentence.
    """
    # 1. Detect N (e.g., if n_gram_counts keys are size 2, then N=3 for Trigrams)
    n = len(list(n_gram_counts.keys())[0]) + 1

    # 2. Iterate through the sentence
    #  start at index 'n-1' so we have enough history
    N = len(sentence)
    total_log_prob = 0
    count = 0

    for i in range(n - 1, N):
        # extract history (previous words) and target (current word)
        history = tuple(sentence[i - (n - 1) : i])
        word = sentence[i]

        # calculate probability
        prob = estimate_probability(
            word,
            history,
            n_gram_counts,
            n_plus1_gram_counts,
            vocab_size,
            k,
        )

        # sum log probabilities
        total_log_prob += math.log(prob)
        count += 1

    # 3. Calculate perplexity
    # exp( -1/N * sum(log_probs) )
    perplexity = math.exp(-total_log_prob / count)
    return perplexity


# --- Test Perplexity Calculation ---
vocab_size = len(vocab)
raw_sent = "The United States are going to war".split()
processed_test = preprocess_sentences([raw_sent], vocab, n=3)[0]

print(f"Processed: {processed_test}")
pp = calculate_perplexity(processed_test, bigrams, trigrams, vocab_size, k=1.0)
print(f"Corrected Perplexity: {pp:.2f}")

Processed: ['<s>', '<s>', 'the', 'united', 'states', 'are', 'going', 'to', 'war', '</s>']
Corrected Perplexity: 1671.39


### 4.5 Model Comparison

In [79]:
def compare_models(test_data, vocab):
    vocab_size = len(vocab)
    print("Counting N-Grams for comparison...")

    # 1. Re-count to be safe
    unigrams = count_n_grams(train_data_processed, 1)
    bigrams = count_n_grams(train_data_processed, 2)
    trigrams = count_n_grams(train_data_processed, 3)

    total_words = sum(unigrams.values())

    print("\n--- Model Comparison (Lower Perplexity is Better) ---")

    # A. Unigram Perplexity (Baseline)
    pp_uni = 0
    c = 0
    for sent in test_data:
        for word in sent:
            if word in ["<s>", "</s>"]:
                continue
            # Probability = (Count + 1) / (Total Words + Vocab Size)
            prob = (unigrams.get((word,), 0) + 1) / (total_words + vocab_size)
            pp_uni += math.log(prob)
            c += 1
    if c > 0:
        print(f"Unigram (N=1): {math.exp(-pp_uni/c):.2f}")

    # Helper for Bigram/Trigram
    def get_avg_pp(n_counts, n_plus1_counts):
        total = 0
        count = 0
        for sent in test_data:
            pp = calculate_perplexity(sent, n_counts, n_plus1_counts, vocab_size, k=1.0)
            if pp < 100000:  # Filter out anomalies
                total += pp
                count += 1
        return total / count if count > 0 else 0

    # B. Bigram
    print(f"Bigram  (N=2): {get_avg_pp(unigrams, bigrams):.2f}")

    # C. Trigram
    print(f"Trigram (N=3): {get_avg_pp(bigrams, trigrams):.2f}")


# --- Run the Final Benchmark ---
compare_models(test_data_processed[:100], vocab)

Counting N-Grams for comparison...

--- Model Comparison (Lower Perplexity is Better) ---
Unigram (N=1): 607.39
Bigram  (N=2): 773.01
Trigram (N=3): 5095.62


With limited training data, we see that the lower n performs better. If we have more data, we should see trigram perform better